# Lab 10: Live Gemini Evaluation

**COMPSS 211A | Fall 2026 | Student copy**

Run a small structured-output call and inspect errors after Monday's LLM lecture.

**Date/deadline:** Friday, November 13, 2026

Work through each task and replace the response placeholders with your own answers.

## Scenario

Make one controlled Gemini request, validate the returned record, and compare it with a saved offline case. The comments are synthetic, and the API key stays outside the notebook.

## Goal

- Keep the key in the environment.
- Validate returned labels.
- Compare against a same-data baseline.

## Keep handy

- **Model:** `gemini-3.6-flash`.
- **Offline copy:** The notebook includes a recorded fixture so ordinary validation never spends money.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import numpy as np
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## Practice: guarded live call and offline fixture

In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"
sample = pd.read_csv(course_data_path("hw4_synthetic_campus_comments.csv")).head(3)
fixture = pd.read_csv(course_data_path("hw5_recorded_evaluation_fixture.csv")).head(3)

def validate_output(document_id, label):
    if label not in {"transit", "study_space", "accessibility", "food", "safety", "services"}:
        raise ValueError(f"Unexpected label for {document_id}: {label}")
    return {"document_id": document_id, "label": label}

def run_live(sample):
    key = os.getenv("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("Set GEMINI_API_KEY in the environment.")
    from google import genai
    client = genai.Client(api_key=key)
    return [
        client.models.generate_content(
            model=GEMINI_MODEL,
            contents=f"Return one routing label for: {row.text}",
        ).text
        for row in sample.itertuples()
    ]

live_enabled = os.getenv("COMPSS211_LIVE_API", "0") == "1"
raw_outputs = (
    run_live(sample)
    if live_enabled
    else fixture["recorded_llm_label"].tolist()
)
outputs = [
    validate_output(document_id, label)
    for document_id, label in zip(sample["document_id"], raw_outputs)
]
display(pd.DataFrame(outputs))

### Your notes

Before you leave, write down one thing you can now do and one question you still have.

> Write your notes here.

## Exit

The last 10 minutes are reserved for the three-question quiz on Monday's material. Use the lab to practice the ideas before you answer from memory.